# PICP-VSign — Full Pipeline: Generate + Train + Evaluate

**Run All — fully automated. Generates dataset, trains contrastive encoder, fine-tunes constraint head, evaluates.**

### Pipeline
1. **Generate** 50,000 radar vital sign samples (~12 min CPU)
2. **Contrastive pretraining** — SimCLR with physics augmentations (200 epochs, ~4h on T4)
3. **Fine-tune** — constrained classification head (100 epochs, ~1.5h on T4)
4. **Evaluate** — baselines + metrics + constraint verification
5. **Save** — models + figures + results

| Component | Detail |
|-----------|--------|
| Radar | 77 GHz, 4TX×4RX MIMO, 512 slow-time samples |
| Dataset | 50,000 samples, 5 body types, HR 40-200 BPM, BR 8-30 BrPM |
| Encoder | ResNet-18 style, 256-dim L2-normalized embeddings |
| Loss | NT-Xent (τ=0.1) + physics-consistent augmentations |
| Decoder | Hard-constrained bins: 161 HR + 45 BR (zero impossible outputs) |
| Target | HR MAE < 3 BPM, BR MAE < 0.8 BrPM |

**Accelerator:** GPU T4 | **Est. total:** ~6 hours

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1: Setup & Dependencies
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, json, datetime, pathlib
import numpy as np

# Install missing packages
for pkg in ['h5py', 'umap-learn']:
    try:
        __import__(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import h5py
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import Counter

# Paths
WORK_DIR = pathlib.Path('/kaggle/working/picp_vsign')
WORK_DIR.mkdir(parents=True, exist_ok=True)
H5_PATH = WORK_DIR / 'vsign_50k.h5'
CKPT_DIR = WORK_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
FIG_DIR = WORK_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# Hardware
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CPUS = mp.cpu_count()
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CPUs: {N_CPUS}')
print(f'PyTorch: {torch.__version__}')
print(f'\n✅ Ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2: Configuration
# ══════════════════════════════════════════════════════════════════════════════

# ─── Dataset ───
N_SAMPLES       = 50000
HR_RANGE        = (40, 200)
BR_RANGE        = (8, 30)
SNR_RANGE_DB    = (5, 40)
BODY_TYPES      = ['thin', 'medium', 'heavy', 'athletic', 'elderly']
PRF             = 51.2
T_SAMPLES       = 512
N_WORKERS       = min(N_CPUS, 4)

# ─── Contrastive Pretraining ───
CL_EPOCHS       = 200
CL_BATCH_SIZE   = 128 if DEVICE.type == 'cuda' else 32
CL_LR           = 3e-4
CL_TEMPERATURE  = 0.1
EMBEDDING_DIM   = 256
PROJECTION_DIM  = 128

# ─── Fine-tuning ───
FT_EPOCHS       = 100
FT_BATCH_SIZE   = 128 if DEVICE.type == 'cuda' else 32
FT_LR           = 1e-4
FREEZE_EPOCHS   = 20

# ─── Constraint Head ───
HR_MIN, HR_MAX, HR_STEP = 40, 200, 1      # 161 bins
BR_MIN, BR_MAX, BR_STEP = 8, 30, 0.5      # 45 bins
N_HR_BINS = int((HR_MAX - HR_MIN) / HR_STEP) + 1
N_BR_BINS = int((BR_MAX - BR_MIN) / BR_STEP) + 1

print(f'Dataset: {N_SAMPLES:,} samples')
print(f'Contrastive: {CL_EPOCHS} epochs, batch={CL_BATCH_SIZE}, τ={CL_TEMPERATURE}')
print(f'Fine-tune: {FT_EPOCHS} epochs, freeze encoder for first {FREEZE_EPOCHS}')
print(f'Constraint bins: HR={N_HR_BINS}, BR={N_BR_BINS}')
print(f'Output: {WORK_DIR}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3: Physics Engine — Radar Vital Sign Simulator
# ══════════════════════════════════════════════════════════════════════════════

# ─── Physiological Motion ─────────────────────────────────────────────────
AMPLITUDES = {
    'thin':     {'hr': 0.0015, 'br': 0.008},
    'medium':   {'hr': 0.002,  'br': 0.010},
    'heavy':    {'hr': 0.001,  'br': 0.015},
    'athletic': {'hr': 0.003,  'br': 0.009},
    'elderly':  {'hr': 0.0012, 'br': 0.007},
}
FAT_THICKNESS = {'thin': 0.010, 'medium': 0.020, 'heavy': 0.040,
                 'athletic': 0.012, 'elderly': 0.025}

def chest_wall_displacement(t, hr_bpm, br_bpm, body_type, phase_hr, phase_br):
    A = AMPLITUDES[body_type]
    omega_hr = 2 * np.pi * hr_bpm / 60.0
    omega_br = 2 * np.pi * br_bpm / 60.0
    d_heart = A['hr'] * (np.sin(omega_hr * t + phase_hr)
              + 0.3 * np.sin(2 * omega_hr * t + phase_hr)
              + 0.1 * np.sin(3 * omega_hr * t + phase_hr))
    d_breath = A['br'] * np.sin(omega_br * t + phase_br)
    return d_heart + d_breath

def get_chest_layers(body_type):
    fat = FAT_THICKNESS[body_type]
    return [
        {'name': 'skin', 'z_start': 0.000, 'z_end': 0.002,
         'eps_inf': 4.0, 'delta_eps': 32.0, 'tau': 7.23e-12, 'alpha': 0.0, 'sigma': 0.0002},
        {'name': 'fat', 'z_start': 0.002, 'z_end': 0.002+fat,
         'eps_inf': 2.5, 'delta_eps': 3.0, 'tau': 7.96e-12, 'alpha': 0.2, 'sigma': 0.01},
        {'name': 'muscle', 'z_start': 0.002+fat, 'z_end': 0.017+fat,
         'eps_inf': 4.0, 'delta_eps': 46.0, 'tau': 7.23e-12, 'alpha': 0.1, 'sigma': 0.7},
        {'name': 'lung', 'z_start': 0.017+fat, 'z_end': 0.120+fat,
         'eps_inf': 1.1, 'delta_eps': 0.5, 'tau': 1.0e-12, 'alpha': 0.0, 'sigma': 0.03},
        {'name': 'heart', 'z_start': 0.050+fat, 'z_end': 0.110+fat,
         'eps_inf': 4.5, 'delta_eps': 46.0, 'tau': 7.96e-12, 'alpha': 0.12, 'sigma': 0.6},
    ]

# ─── Radar Simulation ─────────────────────────────────────────────────────
class VSignRadarSim:
    def __init__(self, layers, freq=77e9, n_tx=4, n_rx=4, spacing=1.948e-3, standoff=0.5):
        self.k0 = 2 * np.pi * freq / 3e8
        self.wavelength = 3e8 / freq
        self.eps0 = 8.854e-12
        self.n_virtual = n_tx * n_rx
        self.layers = layers
        omega = 2 * np.pi * freq
        self.n_layers = []
        for layer in layers:
            eps = (layer['eps_inf'] +
                   layer['delta_eps'] / (1 + (1j * omega * layer['tau'])**(1 - layer['alpha'])))
            eps = eps + layer['sigma'] / (1j * omega * self.eps0)
            self.n_layers.append(np.sqrt(eps))
        self.element_phases = np.zeros(self.n_virtual)
        for tx in range(n_tx):
            for rx in range(n_rx):
                idx = tx * n_rx + rx
                dx = (tx + rx) * spacing
                self.element_phases[idx] = 2*np.pi * dx**2 / (2*standoff) / self.wavelength

    def _reflection(self, displacement):
        r_total = complex(0)
        for i in range(len(self.layers)-1, -1, -1):
            layer = self.layers[i]
            n_i = self.n_layers[i]
            thickness = max(0.001, layer['z_end'] - layer['z_start'])
            n_prev = self.n_layers[i-1] if i > 0 else complex(1.0)
            r_if = (n_prev - n_i) / (n_prev + n_i)
            exp_term = np.exp(-4j * self.k0 * n_i * thickness)
            r_total = (r_if + r_total * exp_term) / (1 + r_if * r_total * exp_term)
        return r_total * np.exp(2j * self.k0 * displacement)

    def snapshot(self, displacement):
        base_r = self._reflection(displacement)
        disp_phase = 4 * np.pi * displacement / self.wavelength
        snap = np.zeros(self.n_virtual, dtype=np.complex64)
        for i in range(self.n_virtual):
            snap[i] = base_r * np.exp(1j * (self.element_phases[i] + disp_phase*(1+0.01*i)))
        return snap

def simulate_one_sample(params):
    hr_bpm, br_bpm, body_type, snr_db, seed = params
    rng = np.random.default_rng(seed)
    radar = VSignRadarSim(get_chest_layers(body_type))
    t_axis = np.linspace(0, 10.0, 512)
    phase_hr = rng.uniform(0, 2*np.pi)
    phase_br = rng.uniform(0, 2*np.pi)
    signals = []
    for t in t_axis:
        d = chest_wall_displacement(t, hr_bpm, br_bpm, body_type, phase_hr, phase_br)
        signals.append(radar.snapshot(d))
    rc = np.stack(signals, axis=-1)  # [16, 512]
    sig_pwr = np.mean(np.abs(rc)**2)
    noise_pwr = sig_pwr / (10**(snr_db/10))
    noise = (rng.normal(0, np.sqrt(noise_pwr/2), rc.shape)
             + 1j*rng.normal(0, np.sqrt(noise_pwr/2), rc.shape))
    rc = (rc + noise).astype(np.complex64)
    return rc, np.array([hr_bpm, br_bpm], dtype=np.float32)

# Quick test
t0 = time.time()
rc_test, vl_test = simulate_one_sample((72.0, 15.0, 'medium', 25.0, 0))
print(f'Single sample: {(time.time()-t0)*1000:.1f}ms')
print(f'Shape: {rc_test.shape}, dtype: {rc_test.dtype}, power: {np.mean(np.abs(rc_test)**2):.4f}')
print(f'✅ Physics engine ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4: Generate 50,000 Samples
# ══════════════════════════════════════════════════════════════════════════════
print(f'Generating {N_SAMPLES:,} samples with {N_WORKERS} workers...')

rng = np.random.default_rng(42)
hr_values = np.arange(HR_RANGE[0], HR_RANGE[1]+1, 1).tolist()
br_values = np.arange(BR_RANGE[0], BR_RANGE[1]+0.5, 0.5).tolist()

params_list = []
for i in range(N_SAMPLES):
    params_list.append((
        float(rng.choice(hr_values)),
        float(rng.choice(br_values)),
        rng.choice(BODY_TYPES),
        float(rng.uniform(*SNR_RANGE_DB)),
        i
    ))

t0 = time.time()
with h5py.File(str(H5_PATH), 'w') as hf:
    radar_ds = hf.create_dataset('radar_cube', shape=(N_SAMPLES, 16, 512),
                                 dtype=np.complex64, chunks=(100, 16, 512))
    label_ds = hf.create_dataset('vital_label', shape=(N_SAMPLES, 2), dtype=np.float32)
    meta_ds  = hf.create_dataset('body_type', shape=(N_SAMPLES,), dtype=h5py.string_dtype())
    snr_ds   = hf.create_dataset('snr_db', shape=(N_SAMPLES,), dtype=np.float32)

    CHUNK = 1000
    done = 0
    for ci in range(0, N_SAMPLES, CHUNK):
        chunk = params_list[ci:ci+CHUNK]
        with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
            results = list(ex.map(simulate_one_sample, chunk))
        for j, (rc, vl) in enumerate(results):
            idx = ci + j
            radar_ds[idx] = rc
            label_ds[idx] = vl
            meta_ds[idx] = chunk[j][2]
            snr_ds[idx] = chunk[j][3]
        done += len(results)
        elapsed = time.time() - t0
        rate = done / elapsed
        eta = (N_SAMPLES - done) / rate
        print(f'  {done:>6,}/{N_SAMPLES:,} | {rate:.0f} samp/s | ETA {eta:.0f}s', end='\r')

elapsed = time.time() - t0
size_gb = H5_PATH.stat().st_size / 1e9
print(f'\n\n✅ Generated {N_SAMPLES:,} samples in {elapsed/60:.1f} min ({size_gb:.2f} GB)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5: Quick Validation + Visual Check
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

with h5py.File(str(H5_PATH), 'r') as hf:
    labels = hf['vital_label'][:]
    n = len(labels)
    bt_arr = [b.decode() for b in hf['body_type'][:]]
    
    # Stats
    print(f'Samples: {n:,}')
    print(f'HR: [{labels[:,0].min():.0f}, {labels[:,0].max():.0f}] BPM')
    print(f'BR: [{labels[:,1].min():.1f}, {labels[:,1].max():.1f}] BrPM')
    bt_counts = Counter(bt_arr)
    for bt in BODY_TYPES:
        print(f'  {bt:>10}: {bt_counts[bt]:,} ({bt_counts[bt]/n*100:.1f}%)')
    
    # FFT check on 5 samples
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    fig.suptitle('Validation: Phase FFT should peak at HR/BR frequencies', fontsize=12)
    freqs = np.fft.rfftfreq(512, d=1.0/PRF)
    
    for col in range(5):
        idx = col * 10000  # Spread across dataset
        rc = hf['radar_cube'][idx]
        hr, br = labels[idx]
        
        # Phase FFT
        phase_sig = np.unwrap(np.angle(rc[0, :]))
        phase_sig -= phase_sig.mean()
        fft_mag = np.abs(np.fft.rfft(phase_sig))
        
        axes[0, col].plot(freqs, fft_mag, 'b-', lw=0.8)
        axes[0, col].axvline(hr/60, color='red', ls='--', alpha=0.8, label=f'HR={hr:.0f}')
        axes[0, col].axvline(br/60, color='green', ls='--', alpha=0.8, label=f'BR={br:.0f}')
        axes[0, col].set_xlim(0, 4)
        axes[0, col].set_title(f'{bt_arr[idx]} HR={hr:.0f} BR={br:.0f}')
        axes[0, col].legend(fontsize=7)
        if col == 0: axes[0, col].set_ylabel('Phase FFT |X(f)|')
        
        # Time domain
        t_ax = np.linspace(0, 10, 512)
        axes[1, col].plot(t_ax[:100], phase_sig[:100], 'b-', lw=0.8)
        axes[1, col].set_xlabel('Time (s)')
        if col == 0: axes[1, col].set_ylabel('Phase (rad)')
    
    plt.tight_layout()
    plt.savefig(str(FIG_DIR / 'validation_fft.png'), dpi=150)
    plt.show()

print('\n✅ Dataset validated — proceeding to training')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6: Model Definitions
# ══════════════════════════════════════════════════════════════════════════════

# ─── Preprocessing ────────────────────────────────────────────────────────
def preprocess_radar_cube(radar_cube):
    '''[B, 16, 512] complex -> [B, 1, 64, 64] float range-Doppler map'''
    B, N_virt, T = radar_cube.shape
    window = torch.hann_window(T, device=radar_cube.device)
    windowed = radar_cube * window.unsqueeze(0).unsqueeze(0)
    doppler = torch.fft.fftshift(torch.fft.fft(windowed, dim=-1), dim=-1)
    center = T // 2
    doppler = doppler[:, :, center-32:center+32]  # [B, 16, 64]
    mag = torch.abs(doppler)
    spatial = torch.abs(torch.fft.fftshift(torch.fft.fft(mag, n=64, dim=1), dim=1))
    spatial = torch.log1p(spatial)
    bmin = spatial.flatten(1).min(1)[0].view(B, 1, 1)
    bmax = spatial.flatten(1).max(1)[0].view(B, 1, 1)
    rdm = (spatial - bmin) / (bmax - bmin + 1e-8)
    return rdm.unsqueeze(1).float()

# ─── Encoder ──────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.GELU(),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c))
        self.skip = (nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                     nn.BatchNorm2d(out_c)) if stride != 1 or in_c != out_c else nn.Identity())
        self.act = nn.GELU()
    def forward(self, x): return self.act(self.conv(x) + self.skip(x))

class VSignEncoder(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU())
        self.layer1 = ResBlock(64, 128, stride=2)
        self.layer2 = ResBlock(128, 256, stride=2)
        self.layer3 = ResBlock(256, 512, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(512, dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer3(self.layer2(self.layer1(x)))
        x = self.pool(x).flatten(1)
        return F.normalize(self.norm(self.proj(x)), dim=-1)

class ProjectionHead(nn.Module):
    def __init__(self, in_dim=256, hidden=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(),
            nn.Linear(hidden, out_dim))
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

# ─── Constraint Head ──────────────────────────────────────────────────────
class ConstrainedVitalHead(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(dim, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 256), nn.GELU())
        self.hr_head = nn.Linear(256, N_HR_BINS)
        self.br_head = nn.Linear(256, N_BR_BINS)
        self.register_buffer('hr_centers', torch.arange(HR_MIN, HR_MAX+HR_STEP, HR_STEP).float())
        self.register_buffer('br_centers', torch.arange(BR_MIN, BR_MAX+BR_STEP, BR_STEP).float())

    def forward(self, emb):
        feat = self.shared(emb)
        hr_logits = self.hr_head(feat)
        br_logits = self.br_head(feat)
        hr_pred = (F.softmax(hr_logits, -1) * self.hr_centers).sum(-1)
        br_pred = (F.softmax(br_logits, -1) * self.br_centers).sum(-1)
        return hr_logits, br_logits, hr_pred, br_pred

# ─── Physics Augmentations ────────────────────────────────────────────────
class PhysicsAugmentor:
    def __call__(self, rc):
        v1, v2 = rc.clone(), rc.clone()
        v1 = self._aug(v1)
        v2 = self._aug(v2)
        return v1, v2
    def _aug(self, rc):
        for _ in range(np.random.randint(2, 4)):
            choice = np.random.randint(4)
            if choice == 0:  # temporal shift
                rc = torch.roll(rc, np.random.randint(-50, 50), dims=-1)
            elif choice == 1:  # noise
                snr = np.random.uniform(5, 40)
                pwr = rc.abs().pow(2).mean() / (10**(snr/10))
                rc = rc + torch.complex(torch.randn_like(rc.real)*pwr.sqrt(),
                                        torch.randn_like(rc.real)*pwr.sqrt())
            elif choice == 2:  # antenna dropout
                mask = (torch.rand(rc.shape[0]) > 0.15).float().unsqueeze(-1)
                rc = rc * mask
            elif choice == 3:  # phase noise
                ph = torch.randn(rc.shape[0]) * (5*np.pi/180)
                rc = rc * torch.exp(1j * ph).unsqueeze(-1)
        return rc

# ─── Losses ───────────────────────────────────────────────────────────────
def nt_xent_loss(z1, z2, temperature=0.1):
    B = z1.shape[0]
    z = torch.cat([z1, z2], 0).float()
    sim = z @ z.T / temperature
    mask = torch.eye(2*B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, -1e9)
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, labels)

def vital_loss(hr_logits, br_logits, hr_pred, br_pred, labels):
    gt_hr, gt_br = labels[:, 0], labels[:, 1]
    hr_bin = ((gt_hr - HR_MIN) / HR_STEP).long().clamp(0, N_HR_BINS-1)
    br_bin = ((gt_br - BR_MIN) / BR_STEP).long().clamp(0, N_BR_BINS-1)
    ce = (F.cross_entropy(hr_logits, hr_bin, label_smoothing=0.1) +
          F.cross_entropy(br_logits, br_bin, label_smoothing=0.1))
    mae = F.l1_loss(hr_pred, gt_hr) + F.l1_loss(br_pred, gt_br)
    return ce + 0.1 * mae

print(f'Encoder params: {sum(p.numel() for p in VSignEncoder().parameters()):,}')
print(f'Head params: {sum(p.numel() for p in ConstrainedVitalHead().parameters()):,}')
print('✅ Models defined')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7: Dataset Class
# ══════════════════════════════════════════════════════════════════════════════

class VSignDataset(Dataset):
    def __init__(self, h5_path, indices=None, augmentor=None):
        self.hf = h5py.File(h5_path, 'r')
        self.radar = self.hf['radar_cube']
        self.labels = self.hf['vital_label'][:]
        self.indices = indices if indices is not None else np.arange(len(self.labels))
        self.augmentor = augmentor

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        rc = torch.from_numpy(self.radar[real_idx].copy())  # [16, 512] complex
        label = torch.from_numpy(self.labels[real_idx])
        if self.augmentor:
            v1, v2 = self.augmentor(rc)
            v1 = preprocess_radar_cube(v1.unsqueeze(0)).squeeze(0)
            v2 = preprocess_radar_cube(v2.unsqueeze(0)).squeeze(0)
            return v1, v2, label
        rdm = preprocess_radar_cube(rc.unsqueeze(0)).squeeze(0)
        return rdm, label

# Split indices
all_idx = np.arange(N_SAMPLES)
np.random.seed(42)
np.random.shuffle(all_idx)
n_train = int(0.8 * N_SAMPLES)
n_val = int(0.1 * N_SAMPLES)
train_idx = all_idx[:n_train]
val_idx = all_idx[n_train:n_train+n_val]
test_idx = all_idx[n_train+n_val:]

print(f'Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')
print('✅ Dataset ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8: Contrastive Pretraining (SimCLR)
# ══════════════════════════════════════════════════════════════════════════════
print(f'═══ Contrastive Pretraining: {CL_EPOCHS} epochs ═══')

augmentor = PhysicsAugmentor()
cl_dataset = VSignDataset(str(H5_PATH), indices=train_idx, augmentor=augmentor)
cl_loader = DataLoader(cl_dataset, batch_size=CL_BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

encoder = VSignEncoder(dim=EMBEDDING_DIM).to(DEVICE)
proj_head = ProjectionHead(EMBEDDING_DIM, 512, PROJECTION_DIM).to(DEVICE)

optimizer = optim.AdamW(list(encoder.parameters()) + list(proj_head.parameters()),
                         lr=CL_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CL_EPOCHS)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

cl_losses = []
t0 = time.time()

for epoch in range(CL_EPOCHS):
    encoder.train(); proj_head.train()
    epoch_loss = 0; n_batch = 0
    
    for v1, v2, _ in cl_loader:
        v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                z1 = proj_head(encoder(v1))
                z2 = proj_head(encoder(v2))
                loss = nt_xent_loss(z1, z2, CL_TEMPERATURE)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(list(encoder.parameters())+list(proj_head.parameters()), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            z1 = proj_head(encoder(v1))
            z2 = proj_head(encoder(v2))
            loss = nt_xent_loss(z1, z2, CL_TEMPERATURE)
            loss.backward()
            nn.utils.clip_grad_norm_(list(encoder.parameters())+list(proj_head.parameters()), 1.0)
            optimizer.step()
        
        epoch_loss += loss.item(); n_batch += 1
    
    scheduler.step()
    avg_loss = epoch_loss / n_batch
    cl_losses.append(avg_loss)
    
    if epoch % 20 == 0 or epoch == CL_EPOCHS - 1:
        elapsed = time.time() - t0
        print(f'  Epoch {epoch:3d}/{CL_EPOCHS}: loss={avg_loss:.4f} | '
              f'lr={scheduler.get_last_lr()[0]:.6f} | {elapsed/60:.1f}min')
    
    if epoch % 50 == 0:
        torch.save(encoder.state_dict(), str(CKPT_DIR / f'encoder_ep{epoch}.pt'))

# Save final
torch.save(encoder.state_dict(), str(CKPT_DIR / 'encoder_pretrained.pt'))
elapsed = time.time() - t0
print(f'\n✅ Contrastive pretraining done in {elapsed/3600:.1f}h')
print(f'   Final loss: {cl_losses[-1]:.4f} (target < 1.0)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9: Fine-Tuning with Constraint Head
# ══════════════════════════════════════════════════════════════════════════════
print(f'═══ Fine-Tuning: {FT_EPOCHS} epochs (freeze encoder first {FREEZE_EPOCHS}) ═══')

ft_train_ds = VSignDataset(str(H5_PATH), indices=train_idx)
ft_val_ds   = VSignDataset(str(H5_PATH), indices=val_idx)
ft_test_ds  = VSignDataset(str(H5_PATH), indices=test_idx)

ft_train_dl = DataLoader(ft_train_ds, batch_size=FT_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
ft_val_dl   = DataLoader(ft_val_ds,   batch_size=512, shuffle=False, num_workers=2)
ft_test_dl  = DataLoader(ft_test_ds,  batch_size=512, shuffle=False, num_workers=2)

# Load pretrained encoder
encoder.load_state_dict(torch.load(str(CKPT_DIR / 'encoder_pretrained.pt'), map_location=DEVICE))
head = ConstrainedVitalHead(dim=EMBEDDING_DIM).to(DEVICE)

best_hr_mae = float('inf')
ft_history = {'hr_mae': [], 'br_mae': [], 'loss': []}
t0 = time.time()

for epoch in range(FT_EPOCHS):
    # Freeze/unfreeze encoder
    encoder.requires_grad_(epoch >= FREEZE_EPOCHS)
    params = list(head.parameters()) + (list(encoder.parameters()) if epoch >= FREEZE_EPOCHS else [])
    optimizer = optim.AdamW(params, lr=FT_LR, weight_decay=1e-4)
    
    # Train
    encoder.train(); head.train()
    epoch_loss = 0; n_batch = 0
    for rdm, label in ft_train_dl:
        rdm, label = rdm.to(DEVICE), label.to(DEVICE)
        emb = encoder(rdm)
        hr_l, br_l, hr_p, br_p = head(emb)
        loss = vital_loss(hr_l, br_l, hr_p, br_p, label)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()
        epoch_loss += loss.item(); n_batch += 1
    
    # Validate
    encoder.eval(); head.eval()
    hr_errs, br_errs = [], []
    with torch.no_grad():
        for rdm, label in ft_val_dl:
            rdm, label = rdm.to(DEVICE), label.to(DEVICE)
            _, _, hr_p, br_p = head(encoder(rdm))
            hr_errs.append((hr_p - label[:, 0]).abs().mean().item())
            br_errs.append((br_p - label[:, 1]).abs().mean().item())
    
    hr_mae = np.mean(hr_errs)
    br_mae = np.mean(br_errs)
    ft_history['hr_mae'].append(hr_mae)
    ft_history['br_mae'].append(br_mae)
    ft_history['loss'].append(epoch_loss / n_batch)
    
    if hr_mae < best_hr_mae:
        best_hr_mae = hr_mae
        torch.save({'encoder': encoder.state_dict(), 'head': head.state_dict(),
                    'epoch': epoch, 'hr_mae': hr_mae, 'br_mae': br_mae},
                   str(CKPT_DIR / 'picp_vsign_best.pt'))
    
    if epoch % 10 == 0 or epoch == FT_EPOCHS - 1:
        marker = ' ★' if hr_mae <= best_hr_mae else ''
        print(f'  Epoch {epoch:3d}: loss={epoch_loss/n_batch:.3f} | '
              f'HR MAE={hr_mae:.2f} BPM | BR MAE={br_mae:.2f} BrPM{marker}')

elapsed = time.time() - t0
print(f'\n✅ Fine-tuning done in {elapsed/3600:.1f}h')
print(f'   Best: HR MAE={best_hr_mae:.2f} BPM')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10: Final Evaluation + Baselines
# ══════════════════════════════════════════════════════════════════════════════
print('═══ Final Evaluation on Test Set ═══\n')

# Load best model
ckpt = torch.load(str(CKPT_DIR / 'picp_vsign_best.pt'), map_location=DEVICE)
encoder.load_state_dict(ckpt['encoder'])
head.load_state_dict(ckpt['head'])
encoder.eval(); head.eval()

# ─── PICP-VSign predictions ───
all_hr_pred, all_br_pred, all_hr_gt, all_br_gt = [], [], [], []
with torch.no_grad():
    for rdm, label in ft_test_dl:
        rdm = rdm.to(DEVICE)
        _, _, hr_p, br_p = head(encoder(rdm))
        all_hr_pred.append(hr_p.cpu().numpy())
        all_br_pred.append(br_p.cpu().numpy())
        all_hr_gt.append(label[:, 0].numpy())
        all_br_gt.append(label[:, 1].numpy())

hr_pred = np.concatenate(all_hr_pred)
br_pred = np.concatenate(all_br_pred)
hr_gt = np.concatenate(all_hr_gt)
br_gt = np.concatenate(all_br_gt)

# ─── Metrics ───
def compute_metrics(pred, gt, name):
    err = pred - gt
    return {
        f'{name}_MAE': np.abs(err).mean(),
        f'{name}_RMSE': np.sqrt((err**2).mean()),
        f'{name}_R2': 1 - np.var(err)/np.var(gt),
        f'{name}_within1': (np.abs(err) < 1).mean()*100,
        f'{name}_within5': (np.abs(err) < 5).mean()*100,
        f'{name}_pct95': np.percentile(np.abs(err), 95),
    }

ours = {**compute_metrics(hr_pred, hr_gt, 'HR'), **compute_metrics(br_pred, br_gt, 'BR')}
ours['HR_impossible'] = ((hr_pred < 40) | (hr_pred > 200)).sum()
ours['BR_impossible'] = ((br_pred < 8) | (br_pred > 30)).sum()

print(f'PICP-VSign (ours):')
print(f'  HR MAE:  {ours["HR_MAE"]:.2f} BPM   (target < 3.0)')
print(f'  BR MAE:  {ours["BR_MAE"]:.2f} BrPM  (target < 0.8)')
print(f'  HR R²:   {ours["HR_R2"]:.4f}')
print(f'  BR R²:   {ours["BR_R2"]:.4f}')
print(f'  HR within 5 BPM:  {ours["HR_within5"]:.1f}%')
print(f'  BR within 1 BrPM: {ours["BR_within1"]:.1f}%')
print(f'  Impossible outputs: HR={ours["HR_impossible"]}, BR={ours["BR_impossible"]}')

# ─── FFT Baseline ───
print(f'\n--- FFT Peak Picking Baseline ---')
with h5py.File(str(H5_PATH), 'r') as hf:
    test_rc = hf['radar_cube'][test_idx[:2000]]  # Subset for speed
    test_labels = hf['vital_label'][test_idx[:2000]]

freqs_fft = np.fft.rfftfreq(512, d=1.0/PRF)
hr_fft, br_fft = np.zeros(2000), np.zeros(2000)
for i in range(2000):
    sig = np.abs(test_rc[i]).mean(axis=0)
    sig = sig - sig.mean()
    fft_m = np.abs(np.fft.rfft(sig))
    hr_mask = (freqs_fft >= 0.67) & (freqs_fft <= 3.33)
    br_mask = (freqs_fft >= 0.13) & (freqs_fft <= 0.5)
    hr_fft[i] = freqs_fft[hr_mask][np.argmax(fft_m[hr_mask])] * 60 if hr_mask.any() else 80
    br_fft[i] = freqs_fft[br_mask][np.argmax(fft_m[br_mask])] * 60 if br_mask.any() else 15

fft_metrics = {**compute_metrics(hr_fft, test_labels[:, 0], 'HR'),
               **compute_metrics(br_fft, test_labels[:, 1], 'BR')}
print(f'  HR MAE: {fft_metrics["HR_MAE"]:.2f} BPM')
print(f'  BR MAE: {fft_metrics["BR_MAE"]:.2f} BrPM')

# ─── Summary Table ───
print(f'\n{"═"*60}')
print(f'{"Method":<25} {"HR MAE (BPM)":<15} {"BR MAE (BrPM)":<15} {"Impossible":<12}')
print(f'{"-"*60}')
print(f'{"FFT Peak Picking":<25} {fft_metrics["HR_MAE"]:<15.2f} {fft_metrics["BR_MAE"]:<15.2f} {"N/A":<12}')
print(f'{"PICP-VSign (ours)":<25} {ours["HR_MAE"]:<15.2f} {ours["BR_MAE"]:<15.2f} {ours["HR_impossible"]+ours["BR_impossible"]:<12}')
print(f'{"═"*60}')

# ─── Constraint verification ───
print(f'\n🔒 HARD CONSTRAINT CHECK:')
print(f'   HR predictions in [{hr_pred.min():.2f}, {hr_pred.max():.2f}] — bounds [40, 200]: '
      f'{"✅ PASS" if hr_pred.min()>=40 and hr_pred.max()<=200 else "❌ FAIL"}')
print(f'   BR predictions in [{br_pred.min():.2f}, {br_pred.max():.2f}] — bounds [8, 30]: '
      f'{"✅ PASS" if br_pred.min()>=8 and br_pred.max()<=30 else "❌ FAIL"}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 11: Publication Figures
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('PICP-VSign Results', fontsize=14, fontweight='bold')

# 1. Training curves
axes[0,0].plot(cl_losses, 'b-', lw=1)
axes[0,0].axhline(1.0, color='r', ls='--', alpha=0.5)
axes[0,0].set(xlabel='Epoch', ylabel='NT-Xent Loss', title='Contrastive Pretraining')
axes[0,0].grid(alpha=0.3)

# 2. Fine-tuning curves
axes[0,1].plot(ft_history['hr_mae'], 'r-', lw=1, label='HR MAE')
axes[0,1].plot(ft_history['br_mae'], 'g-', lw=1, label='BR MAE')
axes[0,1].axhline(3.0, color='r', ls='--', alpha=0.3)
axes[0,1].axhline(0.8, color='g', ls='--', alpha=0.3)
axes[0,1].axvline(FREEZE_EPOCHS, color='gray', ls=':', alpha=0.5, label='Unfreeze')
axes[0,1].set(xlabel='Epoch', ylabel='MAE', title='Fine-tuning')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# 3. HR scatter
axes[0,2].scatter(hr_gt[:2000], hr_pred[:2000], s=3, alpha=0.3, c='tab:blue')
axes[0,2].plot([40, 200], [40, 200], 'r--', lw=1.5)
axes[0,2].set(xlabel='True HR (BPM)', ylabel='Pred HR (BPM)', title=f'HR: MAE={ours["HR_MAE"]:.2f}')
axes[0,2].grid(alpha=0.3)

# 4. BR scatter
axes[1,0].scatter(br_gt[:2000], br_pred[:2000], s=3, alpha=0.3, c='tab:green')
axes[1,0].plot([8, 30], [8, 30], 'r--', lw=1.5)
axes[1,0].set(xlabel='True BR (BrPM)', ylabel='Pred BR (BrPM)', title=f'BR: MAE={ours["BR_MAE"]:.2f}')
axes[1,0].grid(alpha=0.3)

# 5. Error distributions
axes[1,1].hist(hr_pred - hr_gt, bins=80, color='tab:blue', alpha=0.7, density=True)
axes[1,1].axvline(0, color='r', ls='--')
axes[1,1].set(xlabel='HR Error (BPM)', ylabel='Density', title='HR Error Distribution')
axes[1,1].grid(alpha=0.3)

axes[1,2].hist(br_pred - br_gt, bins=80, color='tab:green', alpha=0.7, density=True)
axes[1,2].axvline(0, color='r', ls='--')
axes[1,2].set(xlabel='BR Error (BrPM)', ylabel='Density', title='BR Error Distribution')
axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIG_DIR / 'picp_vsign_results.png'), dpi=200)
plt.show()
print(f'Saved: {FIG_DIR / "picp_vsign_results.png"}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 12: Save Everything + Summary
# ══════════════════════════════════════════════════════════════════════════════

# Save results
results = {
    'timestamp': datetime.datetime.now().isoformat(),
    'dataset': {'n_samples': N_SAMPLES, 'file': str(H5_PATH)},
    'contrastive': {'epochs': CL_EPOCHS, 'final_loss': cl_losses[-1]},
    'finetune': {'epochs': FT_EPOCHS, 'best_epoch': int(ckpt['epoch'])},
    'picp_vsign': ours,
    'fft_baseline': fft_metrics,
    'improvement_hr': fft_metrics['HR_MAE'] - ours['HR_MAE'],
    'improvement_br': fft_metrics['BR_MAE'] - ours['BR_MAE'],
}
with open(str(WORK_DIR / 'results.json'), 'w') as f:
    json.dump(results, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.floating, np.integer)) else str(x))

# Final model
torch.save({'encoder': encoder.state_dict(), 'head': head.state_dict()},
           str(CKPT_DIR / 'picp_vsign_final.pt'))

print('═══════════════════════════════════════════════════════════')
print('  PICP-VSign Pipeline Complete')
print('═══════════════════════════════════════════════════════════')
print(f'  Dataset:     {N_SAMPLES:,} samples ({H5_PATH.stat().st_size/1e9:.2f} GB)')
print(f'  CL Loss:     {cl_losses[-1]:.4f} (target < 1.0)')
print(f'  HR MAE:      {ours["HR_MAE"]:.2f} BPM (target < 3.0)')
print(f'  BR MAE:      {ours["BR_MAE"]:.2f} BrPM (target < 0.8)')
print(f'  HR R²:       {ours["HR_R2"]:.4f} (target > 0.97)')
print(f'  Impossible:  {ours["HR_impossible"] + ours["BR_impossible"]} (target = 0)')
print(f'  vs FFT:      HR {results["improvement_hr"]:.1f} BPM better, BR {results["improvement_br"]:.1f} BrPM better')
print(f'═══════════════════════════════════════════════════════════')
print(f'\nOutput files:')
for f in sorted(WORK_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        label = f'{size/1e6:.1f}MB' if size > 1e6 else f'{size/1e3:.0f}KB'
        print(f'  {f.relative_to(WORK_DIR)} ({label})')
print(f'\n🏁 Done! Download from Kaggle Output tab.')